In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import *
from delta.tables import *
from pyspark.sql.types import *

CATALOG = "hive_streamming"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

DATASET = "test"

BRONZE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.{DATASET}"
SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.{DATASET}"
WATERMARK_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_watermark"
ZORDER_COLS = "viewer_id, filedate" 




In [0]:
def ensure_schemas_and_watermark():

    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")
    


    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {WATERMARK_TABLE} (
      table_name STRING,
      last_processed_ts TIMESTAMP
    )
    USING DELTA
    """)


    spark.sql(f"""
            
              INSERT INTO {WATERMARK_TABLE}
              SELECT '{DATASET}' AS table_name, TIMESTAMP('2025-12-13T00:31:10') AS last_processed_ts
              WHERE NOT EXISTS (
              SELECT 1 FROM {WATERMARK_TABLE} WHERE table_name = '{DATASET}'
              )
             """)
    

In [0]:
def get_last_processed_ts():

    return (spark.table(WATERMARK_TABLE)
            .filter(col("table_name") == DATASET)
            .select("last_processed_ts")
            .collect()[0][0])
    

In [0]:
def create_silver_table_if_missing():
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {SILVER_TABLE} (
            event_id STRING,
            viewer_id STRING,
            snapshot_index INT,
            bitrate_kbps INT,
            bitrate_mbps DOUBLE,
            prev_bitrate_kbps INT,
            quality_switch_flag INT,
            start_buffering_ts TIMESTAMP,
            end_buffering_ts TIMESTAMP,
            is_buffering BOOLEAN,
            buffer_duration_sec DOUBLE,
            ingest_ts TIMESTAMP,
            filename STRING,
            filedate DATE
        )
        USING DELTA
        TBLPROPERTIES (
            delta.enableChangeDataFeed = true,
            delta.autoOptimize.optimizeWrite = true,
            delta.autoOptimize.autoCompact = true
        )
    """)

In [0]:
def reading_incremental_data_from_cdf(last_ts):
      bronze_cdf = (
        spark.read.format("delta")
            .option("readChangeFeed", "true")
            .option("startingTimestamp", str(last_ts))
            .table(BRONZE_TABLE)
    )

      return bronze_cdf.filter(
        col("_change_type").isin("insert", "update_postimage")
       )

In [0]:


def feature_transform(df):

    
    # 1. Standardize column names & types
    
    out = (
        df
        .withColumnRenamed("userID", "viewer_id")
        .withColumnRenamed("snapshotIndex", "snapshot_index")
        .withColumnRenamed("bitrateWatchedKbps", "bitrate_kbps")
        .withColumnRenamed("ingesttime", "ingest_ts")
        .withColumn("snapshot_index", col("snapshot_index").cast("int"))
        .withColumn("bitrate_kbps", col("bitrate_kbps").cast("int"))
    )

    
    # 2. Parse buffering timestamps
    
    out = (
        out
        .withColumn("start_buffering_ts", to_timestamp("startBuffering"))
        .withColumn("end_buffering_ts", to_timestamp("endBuffering"))
        .drop("startBuffering", "endBuffering")
    )

   
    # 3. QoE bitrate features
    
    w_order = Window.partitionBy("viewer_id").orderBy("snapshot_index")

    out = (
        out
        .withColumn("prev_bitrate_kbps", lag("bitrate_kbps").over(w_order))
        .withColumn(
            "quality_switch_flag",
            when(col("prev_bitrate_kbps").isNull(), lit(0))
             .when(col("bitrate_kbps") != col("prev_bitrate_kbps"), lit(1))
             .otherwise(lit(0))
        )
        .withColumn("bitrate_mbps", (col("bitrate_kbps") / lit(1000.0)).cast("double"))
    )

  
    # 4. BUFFER SESSION IDENTIFICATION
    
    out = (
        out
        .withColumn("is_start", col("start_buffering_ts").isNotNull().cast("int"))
        .withColumn("buffer_session_id", sum("is_start").over(w_order))
    )

    w_session = Window.partitionBy("viewer_id", "buffer_session_id")

  
    out = (
        out
        .withColumn(
            "buffer_start_idx",
            min(
                when(col("start_buffering_ts").isNotNull(), col("snapshot_index"))
            ).over(w_session)
        )
        .withColumn(
            "buffer_end_idx",
            min(
                when(col("end_buffering_ts").isNotNull(), col("snapshot_index"))
            ).over(w_session)
        )
    )

    
    # 6. IS_BUFFERING FLAG (state between start & end)
   
    out = (
        out
        .withColumn(
            "is_buffering",
            when(
                (col("buffer_start_idx").isNotNull()) &
                (col("buffer_end_idx").isNotNull()) &
                (col("snapshot_index") >= col("buffer_start_idx")) &
                (col("snapshot_index") <= col("buffer_end_idx")),
                lit(True)
            ).otherwise(lit(False))
        )
    )

   
    # 7. BUFFER DURATION (ONLY ON END ROW)
    
    out = (
        out
        .withColumn(
            "buffer_duration_sec",
            when(
                (col("snapshot_index") == col("buffer_end_idx")) &
                col("start_buffering_ts").isNotNull() == False,  # end row
                (
                    col("end_buffering_ts").cast("long") -
                    min("start_buffering_ts").over(w_session).cast("long")
                ).cast("double")
            ).otherwise(lit(0.0))
        )
    )

    
    # 8. Stable event_id
   
    out = (
        out
        .withColumn(
            "event_id",
            sha2(
                concat_ws(
                    "||",
                    coalesce(col("viewer_id"), lit("")),
                    coalesce(col("snapshot_index").cast("string"), lit("")),
                    coalesce(col("ingest_ts").cast("string"), lit("")),
                    coalesce(col("filename"),lit(""))
                ),
                256
            )
        )
        .dropDuplicates(["event_id"])
    )

    
    # 9. Cleanup helper columns
   
    out = out.drop("is_start", "buffer_start_idx", "buffer_end_idx")

    return out


In [0]:

def merge_into_silver(silver_updates):
  delta_silver = DeltaTable.forName(spark, SILVER_TABLE)

  (
    delta_silver.alias("t")
    .merge(silver_updates.alias("s"), "t.event_id = s.event_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
  )

In [0]:
def capturing_timestamp(silver_updates):
    new_ts = silver_updates.select(max("ingest_ts").alias("mx")).collect()[0]["mx"]
    if new_ts is None:
        return None
      
    spark.sql(f"""
    UPDATE {WATERMARK_TABLE}
    SET last_processed_ts = TIMESTAMP('{new_ts}')
    WHERE table_name = '{DATASET}'
    """)
    return new_ts 

def optimize_silver_table():
  spark.sql(f"""
    OPTIMIZE {SILVER_TABLE}
    ZORDER BY {ZORDER_COLS}
    """)  

In [0]:





  


 


def main():
  ensure_schemas_and_watermark()
  
  
  last_ts = get_last_processed_ts()
  create_silver_table_if_missing()

  bronze_updates = reading_incremental_data_from_cdf(last_ts)

  if bronze_updates.limit(1).count() == 0:
    raise Exception("No new data to process")
  
  values = feature_transform(bronze_updates)

  silver_updates=values.select(
        "event_id",
        "viewer_id",
        "snapshot_index",
        "bitrate_kbps",
        "bitrate_mbps",
        "prev_bitrate_kbps",
        "quality_switch_flag",
        "start_buffering_ts",
        "end_buffering_ts",
        "is_buffering",
        "buffer_duration_sec",
        "ingest_ts",
        "filename",
        "filedate")
  
  silver_updates = silver_updates.cache()
  
  merge_into_silver(silver_updates)

  new_ts = capturing_timestamp(silver_updates)
  
  optimize_silver_table()
  print("SILVER pipline completed")

In [0]:
main()  
